In [41]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np

from neo4j import GraphDatabase

In [ ]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [44]:
cypher = "MATCH (n) RETURN n"
records, summary, keys = driver.execute_query(cypher)

In [45]:
records

[<Record n=<Node element_id='4:8197a03e-5e78-47d7-885e-79bf1e759e75:0' labels=frozenset({'Claim'}) properties={'locution': 'Even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value', 'prediction_index': 0, 'utterance_type': '___Claim___', 'proposition': 'even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value', 'speaker': 'Claire Cooper', 'locution_date': neo4j.time.Date(2020, 5, 28), 'illocutionary_force': 'Asserting', 'uuid': 0}>>,
 <Record n=<Node element_id='4:8197a03e-5e78-47d7-885e-79bf1e759e75:1' labels=frozenset({'Question'}) properties={'locution': 'What can the panel suggest for a whole generation of school children who are now missing nearly half an academic year and face months of distrust abou

In [46]:
for record in records:
    print(record["n"]._properties["prediction_index"])
    break

0


# Ensemble 1: All Results

In [47]:
all_models = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_35_haiku.tsv","anthropic/claude_37_sonnet.tsv", 
          "deepseek/deepseek_v3.tsv", "deepseek/deepseek_r1.tsv",
          "openai/gpt35_turbo.tsv", "openai/gpt4o.tsv", "openai/gpt4o_turbo.tsv", "openai/gpt4o_mini.tsv", "openai/o3_mini.tsv", "openai/gpt45_preview.tsv",
          "vertex/gemini_15_pro.tsv", "vertex/gemini_25_flash.tsv",
          "xai/grok2.tsv", 
          "meta/llama31_8b.tsv", "meta/llama31_405b.tsv", "meta/llama32_3b.tsv", "meta/llama33_70b.tsv", "meta/llama4_maverick.tsv",
          "mistral/mistral_7b.tsv",
          "microsoft/phi4.tsv"]
all_model_names = ["Qwen 3 235b", "Qwen QwQ 32b",
               "Claude 3.5 Haiku", "Claude 3.7 Sonnet", 
               "DeepSeek-V3", "DeepSeek-R1",
               "GPT 3.5 Turbo", "GPT 4o", "GPT 4o Turbo", "GPT 4o Mini", "o3 Mini", "GPT 4.5 Preview",
               "Gemini 1.5 Pro", "Gemini 2.5 Flash",
               "Grok 2", 
               "Llama 3.1:8b", "Llama 3.1:405b", "Llama 3.2:3b", "Llama 3.3:70b", "Llama 4 Maverick",
               "Mistral:7b",
               "Phi 4"]

In [48]:
with open("../../data/processed/PoliticalPositions/ensembles/ensemble1.tsv", "w") as file:
    file.write("nodeID \t array_of_pol_pos \t mean \t var \t std \t NA_count \t prob_of_NA \n")


qwen3_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[0], delimiter="\t")               #"alibaba_cloud/qwen3_235b.tsv"
qwen_qwq_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[1], delimiter="\t")            #"alibaba_cloud/qwen_qwq.tsv"
claude_haiku_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[2], delimiter="\t")        #"anthropic/claude_35_haiku.tsv"
claude_sonnet_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[3], delimiter="\t")       #"anthropic/claude_37_sonnet.tsv", 
deepseek_v3_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[4], delimiter="\t")         #"deepseek/deepseek_v3.tsv"
deepseek_r1_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[5], delimiter="\t")         #"deepseek/deepseek_r1.tsv"
gpt35turbo_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[6], delimiter="\t")          #"openai/gpt35_turbo.tsv"
gpt4o_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[7], delimiter="\t")               #"openai/gpt4o.tsv"
gpt4turbo_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[8], delimiter="\t")           #"openai/gpt4o_turbo.tsv", 
gpt4o_mini_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[9], delimiter="\t")          #"openai/gpt4o_mini.tsv",
o3mini_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[10], delimiter="\t")             #"openai/o3_mini.tsv", 
gpt45_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[11], delimiter="\t")              #"openai/gpt45_preview.tsv",
gemini15_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[12], delimiter="\t")           #"vertex/gemini_15_pro.tsv", 
gemini25_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[13], delimiter="\t")           #"vertex/gemini_25_flash.tsv",
grok2_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[14], delimiter="\t")              #"xai/grok2.tsv", 
llama31_8b_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[15], delimiter="\t")         #"meta/llama31_8b.tsv", 
llama31_405b_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[16], delimiter="\t")       #"meta/llama31_405b.tsv", 
llama32_3b_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[17], delimiter="\t")         #"meta/llama32_3b.tsv", 
llama33_70b_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[18], delimiter="\t")        #"meta/llama33_70b.tsv", 
llama4_mav_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[19], delimiter="\t")         #"meta/llama4_maverick.tsv",
mistral7b_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[20], delimiter="\t")          #"mistral/mistral_7b.tsv",
phi4_df = pd.read_csv("../../data/processed/PoliticalPositions/models/" + all_models[21], delimiter="\t")               #"microsoft/phi4.tsv"

In [49]:
qwen3_df.head()

,nodeID,array_of_pol_pos,mean,var,std,NA_count,prob_of_NA
0,0,"[30, 30, 30, 30]",30.0,0.0,0.0,1,0.2
1,1,[],nan,nan,nan,5,1.0
2,2,[],nan,nan,nan,5,1.0
3,3,[],nan,nan,nan,5,1.0
4,4,"[50, 50]",50.0,0.0,0.0,3,0.6


In [50]:
len(qwen3_df)

23300

In [51]:
eval(qwen3_df.iloc[0][" array_of_pol_pos "])

[30, 30, 30, 30]

In [52]:
int(qwen3_df.iloc[0][" NA_count "])

1

In [53]:
all_models_df = [qwen3_df,qwen_qwq_df,
                 claude_haiku_df,claude_sonnet_df,
                 deepseek_v3_df,deepseek_r1_df,
                 gpt35turbo_df,gpt4o_df,gpt4turbo_df,gpt4o_mini_df,o3mini_df,gpt45_df,
                 gemini15_df,gemini25_df,
                 grok2_df,
                 llama31_8b_df,llama31_405b_df,llama32_3b_df,llama33_70b_df,llama4_mav_df,
                 mistral7b_df,
                 phi4_df]

In [61]:
with open("../../data/processed/PoliticalPositions/ensembles/ensemble1.tsv", "w") as file:
    file.write("nodeID \t array_of_pol_pos \t mean \t var \t std \t NA_count \t prob_of_NA \n")

for record in records:
    i = record["n"]._properties["prediction_index"]
    
    node_level_pol_pos = []
    node_level_na_count = 0
    for model in all_models_df:
        node_level_pol_pos.extend(eval(model.iloc[i][" array_of_pol_pos "]))
        node_level_na_count += int(model.iloc[i][" NA_count "])

    #print(node_level_pol_pos)
    #print(f"{np.mean(node_level_pol_pos):.2f}")
    #print(f"{np.std(node_level_pol_pos):.2f}")
    #print(node_level_na_count)
    #print(f"{node_level_na_count / (len(all_models_df) * 5):.2f}")
    #print(node_level_na_count+len(node_level_pol_pos))
    #print("------------------------")
    with open("../../data/processed/PoliticalPositions/ensembles/ensemble1.tsv", "a") as file:
        file.write(str(i)+" \t "+str(node_level_pol_pos)+" \t "+str(np.mean(node_level_pol_pos))+" \t "+str(np.var(node_level_pol_pos))+" \t "+str(np.std(node_level_pol_pos))+" \t "+str(node_level_na_count)+" \t "+str(node_level_na_count/(len(all_models_df) * 5))+"\n")


## Note to self: remember that results are indexed on 'prediction_index' and not 'uuid' or '<\id\>'

# Ensemble 2: Reasoning Models

In [69]:
reasoning_models = ["alibaba_cloud/qwen3_235b.tsv", "alibaba_cloud/qwen_qwq.tsv",
          "anthropic/claude_37_sonnet.tsv", 
           "deepseek/deepseek_r1.tsv",
          "openai/o3_mini.tsv"]
reasoning_model_names = ["Qwen 3 235b", "Qwen QwQ 32b",
                "Claude 3.7 Sonnet", 
               "DeepSeek-R1",
               "o3 Mini"]

In [70]:
reasoning_models_df = [qwen3_df,qwen_qwq_df,
                 claude_sonnet_df,
                 deepseek_r1_df,
                 o3mini_df]

In [71]:
with open("../../data/processed/PoliticalPositions/ensembles/ensemble2.tsv", "w") as file:
    file.write("nodeID \t array_of_pol_pos \t mean \t var \t std \t NA_count \t prob_of_NA \n")



for record in records:
    i = record["n"]._properties["prediction_index"]
    
    node_level_pol_pos = []
    node_level_na_count = 0
    for model in reasoning_models_df:
        node_level_pol_pos.extend(eval(model.iloc[i][" array_of_pol_pos "]))
        node_level_na_count += int(model.iloc[i][" NA_count "])

    #print(node_level_pol_pos)
    #print(np.mean(node_level_pol_pos))
    #print(np.std(node_level_pol_pos))
    #print(np.var(node_level_pol_pos))
    #print(node_level_na_count)
    #print(node_level_na_count / (len(reasoning_models_df) * 5))
    #print("------------------------")

    with open("../../data/processed/PoliticalPositions/ensembles/ensemble2.tsv", "a") as file:
        file.write(str(i)+" \t "+str(node_level_pol_pos)+" \t "+str(np.mean(node_level_pol_pos))+" \t "+str(np.var(node_level_pol_pos))+" \t "+str(np.std(node_level_pol_pos))+" \t "+str(node_level_na_count)+" \t "+str(node_level_na_count/(len(reasoning_models_df) * 5))+"\n")


# Ensemble 3: Models where |Predictions| > Number of NA scores

In [72]:
ensemble3_models = ["alibaba_cloud/qwen3_235b.tsv",
              "anthropic/claude_37_sonnet.tsv", 
              "deepseek/deepseek_v3.tsv", "deepseek/deepseek_r1.tsv",
              "openai/gpt4o.tsv", "openai/gpt4o_turbo.tsv", "openai/gpt45_preview.tsv",
              "vertex/gemini_15_pro.tsv", "vertex/gemini_25_flash.tsv",
              "xai/grok2.tsv", 
              "meta/llama4_maverick.tsv",
              "microsoft/phi4.tsv"]
ensemble3_model_names = ["Qwen 3 235b",
                   "Claude 3.7 Sonnet", 
                   "DeepSeek-V3", "DeepSeek-R1",
                   "GPT 4o", "GPT 4 Turbo", "GPT 4.5 Preview",
                   "Gemini 1.5 Pro", "Gemini 2.5 Flash",
                   "Grok 2", 
                   "Llama 4 Maverick",
                   "Phi 4"]

In [73]:
ensemble3_models_df = [qwen3_df,
                 claude_sonnet_df,
                 deepseek_v3_df,deepseek_r1_df,
                 gpt4o_df,gpt4turbo_df,gpt45_df,
                 gemini15_df,gemini25_df,
                 grok2_df,
                 llama4_mav_df,
                 phi4_df]

In [74]:
with open("../../data/processed/PoliticalPositions/ensembles/ensemble3.tsv", "w") as file:
    file.write("nodeID \t array_of_pol_pos \t mean \t var \t std \t NA_count \t prob_of_NA \n")


for record in records:
    i = record["n"]._properties["prediction_index"]
    
    node_level_pol_pos = []
    node_level_na_count = 0
    for model in ensemble3_models_df:
        node_level_pol_pos.extend(eval(model.iloc[i][" array_of_pol_pos "]))
        node_level_na_count += int(model.iloc[i][" NA_count "])

    #print(node_level_pol_pos)
    #print(np.mean(node_level_pol_pos))
    #print(np.std(node_level_pol_pos))
    #print(node_level_na_count)
    #print(node_level_na_count / (len(ensemble3_models_df) * 5))
    #print("------------------------")

    with open("../../data/processed/PoliticalPositions/ensembles/ensemble3.tsv", "a") as file:
        file.write(str(i)+" \t "+str(node_level_pol_pos)+" \t "+str(np.mean(node_level_pol_pos))+" \t "+str(np.var(node_level_pol_pos))+" \t "+str(np.std(node_level_pol_pos))+" \t "+str(node_level_na_count)+" \t "+str(node_level_na_count/(len(ensemble3_models_df) * 5))+"\n")
